In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff
import seaborn as sns

import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
# Add the root directory to Python's module search path
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import algorithms.database as database
import algorithms.analysis as analysis
from config import microns_per_pixel, main_image_path, processed_path, live_imaging_path,drugs_image_path
from pathlib import Path



Grid code: splits each embryo into grid regions, analyses properties, and displays them

In [ ]:
nrows = 2
ncolumns = 2
#To make this work for drug/live imaging datasets, need to stop adding the normal dataset to nodes_data
collected_data = []

calc_drugs = False
calc_live = False
show=True

nodes_path = processed_path / "skeleton_networks"
for file_name in os.listdir(nodes_path):
        #Get embryo ID
        embryo_ID, end = file_name.split("_")
        embryo_ID = int(embryo_ID)
        if end=="nodes.csv":
            nodes = pd.read_csv(nodes_path / f"{embryo_ID}_nodes.csv", index_col = 0)
            adj = pd.read_csv(nodes_path / f"{embryo_ID}_adj.csv", index_col = 0)
            adj.columns = adj.columns.astype(int) #Convert headings to ints
            edges = pd.read_csv(nodes_path / f"{embryo_ID}_edges.csv", index_col = 0)
           

            #split into grid regions
            xmin = nodes["x"].quantile(0.01)
            xmax = nodes["x"].quantile(0.99)
            ymax = nodes["y"].quantile(0.99)

            metadata_df = database.initialise_metadata()
            row = metadata_df.loc[embryo_ID]
            #ymin is read from the anterior point
            ymin = row["Anterior_Y"]/microns_per_pixel
            stage = int(row["Stage"])
            n = int(row["n"])
            condition = row["Condition"]
            
            if pd.isna(condition):
                image = cv2.imread(main_image_path / f'hh{stage}_n{n} plain_scaled.jpg')
            else:
                drug = row["Drug"]
                date = row["Experiment_Date"]
                if "live" in drug and calc_live:
                    print("drug")
                    image = cv2.imread(live_imaging_path / f"hh{stage}_n{n}" / f'hh{stage}_n{n}_{condition} plain_scaled.jpg')
                elif not "live" in drug and calc_drugs:
                    image = cv2.imread(drugs_image_path / f"{experiment_date}_{drug}" / f'hh{stage}_n{n}_{condition} plain_scaled.jpg')
                else:
                    continue
            height = len(image)
            width = len(image[0])

            col_lines = np.linspace(xmin,xmax,ncolumns+1)
            row_lines = np.linspace(ymin,ymax,nrows+1)

            neighbours = adj.count()
            nodes["degree"] = neighbours

            if show:
                fig, ax = plt.subplots(figsize=(10, 8))
                #plot grid lines
                ax.hlines(row_lines,0,width)
                ax.vlines(col_lines,0,height)

                #plot iamge
                ax.imshow(image, cmap=plt.cm.gray,alpha=0.5)

                #plot nodes
                scatter = ax.scatter(nodes['x'], nodes['y'], c=nodes['weight'], s=nodes['weight']*3, alpha=0.6, label='Nodes',cmap="winter")
                plt.colorbar(scatter, label="Node Weight")
                ax.get_xaxis().set_visible(False)
                ax.get_yaxis().set_visible(False)

                ax.set_xlim(0,width)
                ax.set_ylim(height,0)
                plt.title(f"Blood Island Size, Stage: HH{stage}, n: {n}")
                plt.show()

            for c in range(ncolumns):
                for r in range(nrows):
                    # Ensure lower bound is strictly smaller than upper bound for .between()
                    x_lower = min(col_lines[c], col_lines[c+1])
                    x_upper = max(col_lines[c], col_lines[c+1])
                    
                    y_lower = min(row_lines[r], row_lines[r+1])
                    y_upper = max(row_lines[r], row_lines[r+1])
                    nodes_region = nodes[nodes["x"].between(x_lower,x_upper) & nodes["y"].between(y_lower,y_upper)]

                   

                    num_nodes = len(nodes_region)
                    # Avoid 'NaN' or warnings if a grid region has absolutely zero nodes
                    avg_degree = np.mean(nodes_region["degree"]) if num_nodes > 0 else 0

                    avg_weight = np.mean(nodes_region["weight"]) if num_nodes > 0 else 0
                    
                    # Append the data as a dictionary
                    collected_data.append({
                        "Embryo_ID": embryo_ID,
                        "Stage": stage,
                        "Condition": condition,
                        "n": n,
                        "Row": r,
                        "Column": c,
                        "Number of Nodes": num_nodes,
                        "Nodes Degree": avg_degree,
                        "Nodes Weight": avg_weight
                    })
#todo: save as csv?
nodes_data=pd.DataFrame(collected_data)



In [ ]:
print(nodes_data)
stages = pd.unique(nodes_data["Stage"].values)

#left_posterior = nodes_data[ (nodes_data["Row"]==0) & (nodes_data["Column"]==0)]
#right_posterior = nodes_data[ (nodes_data["Row"]==0) & (nodes_data["Column"]==1)]
#left_anterior = nodes_data[ (nodes_data["Row"]==1) & (nodes_data["Column"]==0)]
#right_anterior = nodes_data[ (nodes_data["Row"]==1) & (nodes_data["Column"]==1)]

#Relabel the rows and columns to use intuitive string labelling
nodes_data_lab = nodes_data.astype({"Row": str, "Column": str})
nodes_data_lab.loc[nodes_data_lab["Row"]=="0","Row"] = "Anterior"
nodes_data_lab.loc[nodes_data_lab["Row"]=="1","Row"] = "Posterior"
nodes_data_lab.loc[nodes_data_lab["Column"]=="0","Column"] = "Left"
nodes_data_lab.loc[nodes_data_lab["Column"]=="1","Column"] = "Right"

#Seaborn does the heavy lifting of finding the mean Node Degree of each stage, split by rows (hue)
sns.barplot(data=nodes_data_lab, x="Stage",y="Nodes Degree", linewidth=2.5,hue="Row")
plt.xlabel('HH Stage')
plt.ylabel('Mean Node Degree')
plt.title('Mean Blood Island Connectivity by Stage: Anterior vs Posterior')
plt.show()

sns.barplot(data=nodes_data_lab, x="Stage",y="Nodes Degree", linewidth=2.5,hue="Column")
plt.xlabel('HH Stage')
plt.ylabel('Mean Node Degree')
plt.title('Mean Blood Island Connectivity by Stage: Left vs Right')
plt.show()

In [ ]:
sns.barplot(data=nodes_data_lab, x="Stage",y="Nodes Weight", linewidth=2.5,hue="Row")
plt.xlabel('HH Stage')
plt.ylabel('Mean Node Size in $\mu m$')
plt.title('Mean Blood Island Size by Stage: Anterior vs Posterior')
plt.show()

sns.barplot(data=nodes_data_lab, x="Stage",y="Nodes Weight", linewidth=2.5,hue="Column")
plt.xlabel('HH Stage')
plt.ylabel('Mean Node Size in $\mu m$')
plt.title('Mean Blood Island Size by Stage: Left vs Right')
plt.show()

In [ ]:
print(nodes_data_lab)
sns.barplot(data=nodes_data_lab, x="Stage",y="Number of Nodes", linewidth=2.5,hue="Row")
plt.xlabel('HH Stage')
plt.ylabel('Mean Node Size in $\mu m$')
plt.title('Mean Number of Blood Island Clusters by Stage: Anterior vs Posterior')
plt.show()

sns.barplot(data=nodes_data_lab, x="Stage",y="Number of Nodes", linewidth=2.5,hue="Column")
plt.xlabel('HH Stage')
plt.ylabel('Mean Node Size in $\mu m$')
plt.title('Mean Number of Blood Island Clusters by Stage: Left vs Right')
plt.show()

In [ ]:
#Old code: not working: designed for n>2 rows
col_labels = ["Left", "Right"]
row_labels = ["Extreme Anterior", "Central Anterior","Central Posterior", "Extreme Posterior"]
mean_weights = pd.DataFrame(columns=row_labels,index=stages)

for i in stages:
    all_row_data = []
    for j in range(nrows):
        stage_data = nodes_data[nodes_data["Stage"]==i]

        current_row = stage_data[ (stage_data["Row"]==j)]

        mean_weight_current = current_row["Nodes Weight"].mean()

        label = row_labels[j]

        mean_weights.loc[i,label] = mean_weight_current
        #mean_weight_posterior.append(posterior["Nodes Weight"].mean())

x = np.arange(len(stages))
width = 0.35 
positions = np.linspace(x-width/2,x+width/2,nrows) 
print(positions)
colours = ["darkblue","skyblue","coral","maroon"]
for j in range(nrows):
    label = row_labels[j]
    plt.bar(positions[j], mean_weights[label], width/(nrows-1), label=label, color=colours[j])


# 3. Plot both sets of bars, shifting them left and right of the center point
# x - width/2 shifts it left, x + width/2 shifts it right
#plt.bar(x-width/2, mean_weight_anterior, width, label='Anterior', color='skyblue')
#plt.bar(x+width/2, mean_weight_posterior, width, label='Posterior', color='salmon')
plt.xlabel('HH Stage')
plt.ylabel('Node Size in $\mu m$')
plt.title('Mean Blood Island Size by Stage: Anterior vs Posterior')
plt.xticks(x, stages)
plt.legend()